<a href="https://colab.research.google.com/github/elangbijak4/multi-agent-AI/blob/main/Hyperparameter_Tuning4_Agent_perbaikan_dead_lock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# GRID WORLD INTELLIGENT AGENT
# Multi-Brain + Hyperparameter-Tuned Agent
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

# ------------------------------------------------------------
# PERBAIKAN 1 DEAD-LOCK
# ------------------------------------------------------------
from collections import deque

VISIT_MEMORY = deque(maxlen=15)   # short-term memory

# ------------------------------------------------------------
# 1. GRID WORLD ENVIRONMENT
# ------------------------------------------------------------
GRID_SIZE = 10
N_DIRT = 12
N_BLOCK = 10

EMPTY = 0
DIRT = 1
BLOCK = -1
AGENT = 2

np.random.seed(1)

grid = np.zeros((GRID_SIZE, GRID_SIZE), dtype=int)

# Place dirt
for _ in range(N_DIRT):
    x, y = np.random.randint(0, GRID_SIZE, 2)
    grid[x, y] = DIRT

# Place blocks
for _ in range(N_BLOCK):
    x, y = np.random.randint(0, GRID_SIZE, 2)
    if grid[x, y] == EMPTY:
        grid[x, y] = BLOCK

# Agent start
agent_pos = [0, 0]

# ------------------------------------------------------------
# 2. ACTION SPACE
# ------------------------------------------------------------
ACTIONS = {
    0: (-1, 0),  # UP
    1: (1, 0),   # DOWN
    2: (0, -1),  # LEFT
    3: (0, 1)    # RIGHT
}

# ------------------------------------------------------------
# 3. PERCEPTION MODEL
# ------------------------------------------------------------
def perceive(grid, pos):
    """
    Local perception: 4-neighborhood
    """
    x, y = pos
    obs = []
    for dx, dy in ACTIONS.values():
        nx, ny = x + dx, y + dy
        if nx < 0 or ny < 0 or nx >= GRID_SIZE or ny >= GRID_SIZE:
            obs.append(BLOCK)
        else:
            obs.append(grid[nx, ny])
    return obs

# ------------------------------------------------------------
# 4. TRAINING DATA (Synthetic World Knowledge)
# ------------------------------------------------------------
X_train = []
y_train = []

for _ in range(600):
    obs = np.random.choice([EMPTY, DIRT, BLOCK], size=4)

    # Heuristic teacher:
    if DIRT in obs:
        action = np.where(obs == DIRT)[0][0]
    else:
        safe = [i for i, v in enumerate(obs) if v != BLOCK]
        action = np.random.choice(safe) if safe else 0

    X_train.append(obs)
    y_train.append(action)

X_train = np.array(X_train)
y_train = np.array(y_train)

# ------------------------------------------------------------
# 5. BRAIN FACTORY (HYPERPARAMETERS FIXED FROM TUNING)
# ------------------------------------------------------------
brains = {
    "Tree": DecisionTreeClassifier(max_depth=5),
    "Bayes": GaussianNB(),
    "Linear": LogisticRegression(max_iter=1000),
    "SVM": SVC(probability=True)
}

for brain in brains.values():
    brain.fit(X_train, y_train)

ACTIVE_BRAIN = brains["Tree"]  # <- change to demo different brains

# ------------------------------------------------------------
# 6. AGENT STEP FUNCTION
# ------------------------------------------------------------
#def step(grid, pos):
#    obs = perceive(grid, pos)
#    action = ACTIVE_BRAIN.predict([obs])[0]

#    dx, dy = ACTIONS[action]
#    nx, ny = pos[0] + dx, pos[1] + dy

#    if nx < 0 or ny < 0 or nx >= GRID_SIZE or ny >= GRID_SIZE:
#        return pos

#    if grid[nx, ny] == BLOCK:
#        return pos

#    if grid[nx, ny] == DIRT:
#        grid[nx, ny] = EMPTY

#    return [nx, ny]

# ------------------------------------------------------------
# PERBAIKAN 2 DEAD-LOCK
# ------------------------------------------------------------
def step(grid, pos, epsilon=0.15):
    """
    Agent step with anti-deadlock mechanism
    """
    obs = perceive(grid, pos)
    state_signature = tuple(pos + obs) # Removed .tolist()

    # Record state visit
    VISIT_MEMORY.append(state_signature)

    # --------------------------
    # Check deadlock condition
    # --------------------------
    deadlock = VISIT_MEMORY.count(state_signature) > 2

    # --------------------------
    # Decide action
    # --------------------------
    if deadlock or np.random.rand() < epsilon:
        # Exploratory escape
        safe_actions = []
        for a, (dx, dy) in ACTIONS.items():
            nx, ny = pos[0] + dx, pos[1] + dy
            if 0 <= nx < GRID_SIZE and 0 <= ny < GRID_SIZE:
                if grid[nx, ny] != BLOCK:
                    safe_actions.append(a)

        action = np.random.choice(safe_actions) if safe_actions else 0
    else:
        # Normal brain decision
        action = int(ACTIVE_BRAIN.predict([obs])[0])

    # --------------------------
    # Execute action
    # --------------------------
    dx, dy = ACTIONS[action]
    nx, ny = pos[0] + dx, pos[1] + dy

    if nx < 0 or ny < 0 or nx >= GRID_SIZE or ny >= GRID_SIZE:
        return pos

    if grid[nx, ny] == BLOCK:
        return pos

    if grid[nx, ny] == DIRT:
        grid[nx, ny] = EMPTY

    return [nx, ny]

# ------------------------------------------------------------
# 7. ANIMATION
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5,5))
im = ax.imshow(grid, cmap="viridis", vmin=-1, vmax=2)

def update(frame):
    global agent_pos, grid
    agent_pos = step(grid, agent_pos)

    display = grid.copy()
    display[agent_pos[0], agent_pos[1]] = AGENT
    im.set_data(display)
    ax.set_title(f"Step {frame}")
    return [im]

ani = animation.FuncAnimation(
    fig, update, frames=80, interval=300
)

plt.close()
HTML(ani.to_jshtml())